In [0]:
%run ./01_project_config

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
01 - PROJECT CONFIGURATION
CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
CAMARA_API_BASE_URL: https://dadosabertos.camara.leg.br/api/v2
RUN_ID: dcd1d698-0822-4797-9cbb-1e05378ff076
PROJECT CONFIGURATION LOADED SUCCESSFULLY


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 91 Reset Project Environment
# MAGIC
# MAGIC Resets the Brazil Legislative Analytics Medallion environment by removing tables and schemas created during development and testing.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook supports environment cleanup during development iterations, architecture changes and integration tests.
# MAGIC
# MAGIC ## Architecture
# MAGIC The project uses a simplified Medallion Architecture:
# MAGIC
# MAGIC Bronze → Silver → Gold → Marts
# MAGIC
# MAGIC ## Actions Performed
# MAGIC - Drops audit tables
# MAGIC - Drops Bronze tables
# MAGIC - Drops Silver tables
# MAGIC - Drops Gold tables
# MAGIC - Drops Mart tables
# MAGIC - Optionally removes schemas
# MAGIC
# MAGIC ## Important Notes
# MAGIC - This notebook is intended for development environments only.
# MAGIC - This notebook permanently removes project objects.
# MAGIC - Execute with caution.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ./01_project_config

# COMMAND ----------

from datetime import datetime

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("91 - RESET PROJECT ENVIRONMENT")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Environment: {PROJECT_ENVIRONMENT}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# RESET CONFIGURATION
# ============================================================
#
# True  -> remove schemas after dropping tables
# False -> keep schemas and remove only tables
#
# ============================================================

DROP_SCHEMAS = False

# COMMAND ----------

# ============================================================
# SCHEMAS
# ============================================================

SCHEMAS_TO_RESET = [
    SCHEMA_AUDIT,
    SCHEMA_BRONZE,
    SCHEMA_SILVER,
    SCHEMA_GOLD,
    SCHEMA_MARTS,
]

# COMMAND ----------

# ============================================================
# TABLE COLLECTIONS
# ============================================================

AUDIT_TABLES = [
    AUD_TB_LOG_EXECUCAO_PIPELINE,
    AUD_TB_LOG_ERROS_PIPELINE,
    AUD_TB_LOG_QUALIDADE_DADOS,
]

BRONZE_TABLE_LIST = list(BRONZE_TABLES.values())
SILVER_TABLE_LIST = list(SILVER_TABLES.values())
GOLD_DIMENSION_TABLE_LIST = list(GOLD_DIMENSION_TABLES.values())
GOLD_FACT_TABLE_LIST = list(GOLD_FACT_TABLES.values())
MART_TABLE_LIST = list(MART_TABLES.values())

# COMMAND ----------

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def drop_table(
    schema_name: str,
    table_name: str,
) -> None:
    """
    Drops a table if it exists.
    """

    full_table_name = (
        f"{CATALOG_NAME}."
        f"{schema_name}."
        f"{table_name}"
    )

    spark.sql(f"""
    DROP TABLE IF EXISTS {full_table_name}
    """)

    print(f"Table removed: {full_table_name}")

# COMMAND ----------

def drop_schema(
    schema_name: str,
) -> None:
    """
    Drops a schema if it exists.
    """

    full_schema_name = (
        f"{CATALOG_NAME}."
        f"{schema_name}"
    )

    spark.sql(f"""
    DROP SCHEMA IF EXISTS {full_schema_name}
    """)

    print(f"Schema removed: {full_schema_name}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Remove Audit Tables

# COMMAND ----------

for table_name in AUDIT_TABLES:
    drop_table(
        schema_name=SCHEMA_AUDIT,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Remove Bronze Tables

# COMMAND ----------

for table_name in BRONZE_TABLE_LIST:
    drop_table(
        schema_name=SCHEMA_BRONZE,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Remove Silver Tables

# COMMAND ----------

for table_name in SILVER_TABLE_LIST:
    drop_table(
        schema_name=SCHEMA_SILVER,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Remove Gold Dimension Tables

# COMMAND ----------

for table_name in GOLD_DIMENSION_TABLE_LIST:
    drop_table(
        schema_name=SCHEMA_GOLD,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Remove Gold Fact Tables

# COMMAND ----------

for table_name in GOLD_FACT_TABLE_LIST:
    drop_table(
        schema_name=SCHEMA_GOLD,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Remove Analytical Mart Tables

# COMMAND ----------

for table_name in MART_TABLE_LIST:
    drop_table(
        schema_name=SCHEMA_MARTS,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Optional Schema Removal

# COMMAND ----------

if DROP_SCHEMAS:

    print("=" * 90)
    print("REMOVING SCHEMAS")
    print("=" * 90)

    for schema_name in SCHEMAS_TO_RESET:
        drop_schema(
            schema_name=schema_name,
        )

else:

    print("=" * 90)
    print("SCHEMA REMOVAL DISABLED")
    print("=" * 90)
    print("Schemas were preserved.")
    print("Only tables were removed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Validate Remaining Structures

# COMMAND ----------

remaining_schemas_df = spark.sql(f"""
SHOW SCHEMAS IN {CATALOG_NAME}
""")

display(remaining_schemas_df)

# COMMAND ----------

# ============================================================
# RESET SUMMARY
# ============================================================

print("=" * 90)
print("RESET SUMMARY")
print("=" * 90)
print(f"Catalog: {CATALOG_NAME}")
print(f"Drop Schemas Enabled: {DROP_SCHEMAS}")

print("=" * 90)
print("RESETTED SCHEMAS")

for schema_name in SCHEMAS_TO_RESET:
    print(f"{CATALOG_NAME}.{schema_name}")

print("=" * 90)
print("PROJECT ENVIRONMENT RESET COMPLETED")
print("=" * 90)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Governance Notes
# MAGIC
# MAGIC This notebook supports environment cleanup for:
# MAGIC
# MAGIC - architecture refactoring
# MAGIC - integration testing
# MAGIC - development iterations
# MAGIC - Medallion restructuring
# MAGIC
# MAGIC ## Safety Recommendation
# MAGIC
# MAGIC Recommended configuration:
# MAGIC
# MAGIC ```python
# MAGIC DROP_SCHEMAS = False
# MAGIC ```
# MAGIC
# MAGIC This preserves governance structures while removing only tables.
# MAGIC
# MAGIC ## Next Step
# MAGIC Recreate the environment by executing:
# MAGIC
# MAGIC ```text
# MAGIC 00_create_catalog_schemas
# MAGIC 01_project_config
# MAGIC 02_audit_tables
# MAGIC 90_validate_project_setup
# MAGIC ```

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
91 - RESET PROJECT ENVIRONMENT
Execution Timestamp: 2026-05-20 02:34:29.857940
Catalog: brazil_legislative_analytics
Environment: dev
Table removed: brazil_legislative_analytics.audit.aud_log_execucao_pipeline
Table removed: brazil_legislative_analytics.audit.aud_log_erros_pipeline
Table removed: brazil_legislative_analytics.audit.aud_log_qualidade_dados
Table removed: brazil_legislative_analytics.bronze.br_deputados
Table removed: brazil_legislative_analytics.bronze.br_frentes
Table removed: brazil_legislative_analytics.bronze.br_frentes_membros
Table removed: brazil_legislative_analytics.bronze.br_frentes_detalhes
Table removed: brazil_legislative_analytics.bronze.br_eventos
Table removed: brazil_legislative_analytics.bronze.br_votacoes
Table removed: brazil_legislative_analytics.bronze.br_votos
Table removed: brazil_legislative_analytics.bronze.br_despesas_ceap
Table removed: brazil_legislative_analytics.bronze.br_cpis
Table removed: brazil_leg

databaseName
audit
bronze
gold
information_schema
marts
silver


RESET SUMMARY
Catalog: brazil_legislative_analytics
Drop Schemas Enabled: False
RESETTED SCHEMAS
brazil_legislative_analytics.audit
brazil_legislative_analytics.bronze
brazil_legislative_analytics.silver
brazil_legislative_analytics.gold
brazil_legislative_analytics.marts
PROJECT ENVIRONMENT RESET COMPLETED
